# 2 — Raw Activation Extraction (Objects + Checkers)

**Purpose:** Extract raw activation statistics (`activation_sum`, `firing_count`) for both object prompts and checker prompts. These raw stats can be re-thresholded at any epsilon/consistency combination without re-running the model.

In [ ]:
# Cell 1 – Dependency setup
import subprocess, sys, os

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "accelerate"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Dependencies ready")

In [ ]:
# Cell 2 – Configuration
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
MODEL_ID = MODEL_CONFIGS[MODEL]["id"]

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
OBJECT_PROMPTS = f"{PREFIX}1A_object_prompts.parquet"
CHECKER_PROMPTS = f"{PREFIX}1B_checker_prompts.parquet"
OBJECT_OUTPUT = f"{PREFIX}2_object_activations.h5"
CHECKER_OUTPUT = f"{PREFIX}2_checker_activations.h5"
N_LAYERS = MODEL_CONFIGS[MODEL]["n_layers"]
CHECKPOINT_EVERY = 200

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/New-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/New-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")
print(f"Object prompts: {OBJECT_PROMPTS}")
print(f"Checker prompts: {CHECKER_PROMPTS}")

In [ ]:
# Cell 3 – Imports
import numpy as np, pandas as pd, torch, h5py
from tqdm.auto import tqdm
import logging, re

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("notebook12")
print("Imports OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Cell 4 – Load model & tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()

print(f"Model loaded | Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 5 – Register hooks
from module2.extraction import ActivationExtractor

mlp_pattern = None
seen_lids = set()
for name, _ in model.named_modules():
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        lid = int(m.group(2))
        seen_lids.add(lid)
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)

print(f"Pattern: {mlp_pattern} | Layers: {sorted(seen_lids)}")

extractor = ActivationExtractor(
    model=model, tokenizer=tokenizer, device=DEVICE,
    n_layers=N_LAYERS, use_hook_recorder=False,
)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()

# Sanity check
acts = extractor.extract('x = 42')
for lid, vec in sorted(acts.items()):
    print(f"  Layer {lid}: shape={tuple(vec.shape)}, mean |act|={vec.abs().mean().item():.6f}")

In [ ]:
# Cell 6 – Extract OBJECT activations
from module2.binarization import RawActivationCollector
from module2.io_utils import save_activations_hdf5
import pickle

df_obj = pd.read_parquet(os.path.join(DATA_DIR, OBJECT_PROMPTS))
print(f"Object prompts: {len(df_obj)} rows, {df_obj.groupby(['ast_node','builtin_obj']).ngroups} pairs")

collector = RawActivationCollector(n_layers=N_LAYERS)
grouped = df_obj.groupby(["ast_node", "builtin_obj"])
all_pairs = list(grouped.groups.keys())

pair_activations = {}
for idx, (ast_n, blt_o) in enumerate(tqdm(all_pairs, desc="Object extraction")):
    prompts = df_obj.loc[grouped.groups[(ast_n, blt_o)], "prompt_text"].tolist()
    raw = collector.collect(extractor, prompts)
    pair_activations[(ast_n, blt_o)] = raw

# Save
obj_output_path = os.path.join(DATA_DIR, OBJECT_OUTPUT)
metadata = {
    "model_id": MODEL_ID,
    "n_layers": N_LAYERS,
    "n_pairs": len(pair_activations),
    "input_file": OBJECT_PROMPTS,
    "ast_nodes": sorted(set(a for a, _ in pair_activations)),
    "builtin_objs": sorted(set(b for _, b in pair_activations)),
}
save_activations_hdf5(obj_output_path, pair_activations, metadata)
print(f"Saved: {obj_output_path}")

In [ ]:
# Cell 7 – Extract CHECKER activations
df_chk = pd.read_parquet(os.path.join(DATA_DIR, CHECKER_PROMPTS))
print(f"Checker prompts: {len(df_chk)} rows, {df_chk['object'].nunique()} objects")

checker_activations = {}
for obj_key in tqdm(sorted(df_chk["object"].unique()), desc="Checker extraction"):
    prompts = df_chk[df_chk["object"] == obj_key]["prompt_text"].tolist()
    if len(prompts) == 0:
        continue
    raw = collector.collect(extractor, prompts)
    # Split obj_key on __ to create tuple key for save_activations_hdf5 compatibility
    parts = obj_key.split("__", 1)
    checker_activations[(parts[0], parts[1])] = raw

# Save
chk_output_path = os.path.join(DATA_DIR, CHECKER_OUTPUT)
chk_metadata = {
    "model_id": MODEL_ID,
    "n_layers": N_LAYERS,
    "n_pairs": len(checker_activations),
    "input_file": CHECKER_PROMPTS,
}
save_activations_hdf5(chk_output_path, checker_activations, chk_metadata)
print(f"Saved: {chk_output_path}")

In [ ]:
# Cell 8 – Verification
for label, path in [("Object", obj_output_path), ("Checker", chk_output_path)]:
    with h5py.File(path, "r") as f:
        counts = [0]
        def count(name, obj):
            if isinstance(obj, h5py.Dataset):
                counts[0] += 1
        f.visititems(count)
        print(f"{label}: {path}")
        print(f"  Datasets: {counts[0]}")

In [ ]:
# Cell 9 – Cleanup
extractor.remove_hooks()
print("Hooks removed.")
print(f"\n12 complete.")
print(f"  Object activations: {obj_output_path}")
print(f"  Checker activations: {chk_output_path}")